In [17]:
import cv2
import os
import shutil
from pathlib import Path
from functools import partial
import matplotlib.pyplot as plt
from collections import defaultdict
import math
import shutil
import numpy as np
import re
import ntpath
import random

The main goal is to generate a dataset for classification of left right eye.

We use the source `classification_source_dataset.txt` which is the fusion of pngs.txt and pngs2.txt.

In [11]:
def join_dataset(dataset1_file: str, dataset2_file: str, output_file: str):
    def strip(str_: str):
        return str_.strip()
    
    def is_exist(file):
        return Path(file).exists()
    
    if not Path(dataset1_file).exists() or not Path(dataset2_file).exists():
        return
        
    with open(dataset1_file, "r") as file1, \
         open(dataset2_file, "r") as file2, \
         open(output_file, "w") as output:
        
        lst1 = map(strip,file1.readlines())
        lst2 = map(strip,file2.readlines())
        
        # remove duplicate element
        s1 = set(lst1)
        s2 = set(lst2)
        mult_set = s1 | s2
                
        lst1 = sorted(mult_set)
        
        res = list(filter(is_exist, lst1))
        print(len(res))
        
        output.writelines(line + "\n" for line in res)
        
        
join_dataset("source/pngs.txt", "source/pngs2.txt", "source/classification_source_dataset.txt")

We have to split the dataset into 3 sub dataset (train, valid and test).
You can choose the repartition.

An heristic is:
- Subject more represented in the dataset should more be in the train and the test should contain a diversity of subject.

In [12]:
def extract_subject_name(file: str):
    file = file.strip()
    
    # from a path get the basename
    filename = ntpath.basename(file)
    
    lst = filename.split('_')
    if len(lst) == 0:
        return
    
    subject_id = lst[1]
    
    # get the biggest substring of char
    sequences = re.findall(r'\D+', subject_id)
    return max(sequences, key=len, default="")

def get_histogram(files: list[str]) -> dict:
    """
        Do an histogram of name with the name in key and the number of eye with this name
    """
    hist = {}
    
    for file in files:
        name = extract_subject_name(file)
        
        count = hist.get(name, 0)
        hist[name] = count + 1
    
    return hist

Algo's main part

In [13]:
def _fill_classes(rng, hist: dict,
                values: np.ndarray, weights: np.ndarray, total_images: int, 
                quantity_goal: int, interval: int, timeout = 2000):
    """
        use a timetout because greedy doesn't garanty a global convergence
    """
    
    values = values.copy()
    weights = weights.copy()
    
    result = []    
    count_result = 0
    invalid_draw = 0
    
    while invalid_draw < timeout and values.size > 0:
        # normalisation (garenti that the sum is 1)
        p = weights / weights.sum()
        idx = rng.choice(values.size, p=p)
        
        draw = values[idx]
        ratio = ((count_result + hist[draw]) / total_images) - quantity_goal
        
        # if we add the patient we add is to big we look for a better one
        if ratio > interval:
            invalid_draw += 1
            continue
        
        invalid_draw = 0
        result.append(draw)
        count_result += hist[draw]

        # remove
        values = np.delete(values, idx)
        weights = np.delete(weights, idx)

        # => 0 <= ratio < interval
        if ratio >= 0:
            break
            
    return result, count_result, values

def probabilistic_greedy_function(train, ratio_train: int, 
                                  valid, ratio_valid: int,
                                  test, ratio_test: int,
                                  lambda_size = 10, lambda_diversity = 1):
    
    # E_size    
    [_, n_image_train] = train
    [_, n_image_valid] = valid
    [subjects_test, n_image_test] = test
    
    n_image = n_image_train + n_image_test + n_image_valid

    E_size = (abs((n_image_train / n_image) - ratio_train)
              + abs((n_image_valid / n_image) - ratio_valid)
              + abs((n_image_test / n_image) - ratio_test))
    
    # E_diversity
    E_diversity = len(subjects_test) / n_image_test if n_image_test > 0 else 0.0
    
    return lambda_size * E_size - lambda_diversity * E_diversity

def repeated_weighted_draw (hist: dict[str, int],
                           ratio_train: float, ratio_test: float, 
                           itrain: float, itest: float,
                           draw_nb = 1000):
    """
    :params qtrain, qvalid, qtest: 
        are the quantity of the dataset in each category and
    :params itrain, itest:
        are the degree of liberty around the quantity's goal
    """
        
    rng = np.random.default_rng()
    
    best = None
    best_cost = np.inf
    
    for _ in range(draw_nb):
        subjects = np.asarray(list(hist.keys()))
        weights = np.asarray(list(hist.values()))
        
        total_images_nb = weights.sum()
        
        # draw for train
        train, train_images_nb, remaining = _fill_classes(rng, hist, subjects, weights, 
                                                          total_images_nb, ratio_train, itrain)
        
        subjects = remaining.copy()
        weights = np.asarray([hist[s] for s in subjects])
        
        # draw for test
        test, test_images_nb, remaining = _fill_classes(rng, hist, subjects, 1.0 / weights, 
                                                        total_images_nb, ratio_test, itest)
        
        # the remaining goes to valid
        valid = remaining.copy()
        ratio_valid = 1 - ratio_train - ratio_test
        valid_images_nb = total_images_nb - train_images_nb - test_images_nb
        
        # eval the quality of the split
        cost = probabilistic_greedy_function([train, train_images_nb], ratio_train,
                                                [valid, valid_images_nb], ratio_valid,
                                                [test, test_images_nb], ratio_test)
        
        # maximise the quality
        if cost < best_cost:
            best = ([train, train_images_nb],
                    [valid, valid_images_nb],
                    [test, test_images_nb])
            best_cost = cost
        
    return best

In [14]:
def _log_subject_eye(train, valid, test):
    
    subject_train, nb_image_train = train
    subject_valid, nb_image_valid = valid
    subject_test, nb_image_test = test
    
    nb_subject_train = len(subject_train)
    nb_subject_valid = len(subject_valid)
    nb_subject_test = len(subject_test)
    
    total_subject = nb_subject_train + nb_subject_valid + nb_subject_test
    total_eyes = nb_image_train + nb_image_valid + nb_image_test

    s =  f"train : subject {nb_subject_train}, eye {nb_image_train}\n"
    s += f"valid : subject {nb_subject_valid}, eye {nb_image_valid}\n"
    s += f"test  : subject {nb_subject_test}, eye {nb_image_test}\n"
    s += f"total : subject {total_subject}, eye {total_eyes}\n"

    print(s)

def _fill_file(images: list[str], output_file: str, subdataset: np.ndarray[str]):
    
    def find_name(subject, filename):
        basename = ntpath.basename(filename)
        return re.search(subject, basename) is not None
    
    with open(output_file, "w") as file:
        
        for subject in subdataset:
            subject_files = list(filter(partial(find_name, subject), images))
            file.writelines(subject_files)

def split_dataset_source(dataset: str, output_dir: str,
                  ratio_train = 0.70, ratio_valid = 0.15, ratio_test = 0.15, 
                  itrain = 0.05, itest = 0.02):
    
    if not Path(dataset).exists():
        return
    
    with open(dataset, "r") as file:
        images = file.readlines()
        
        hist = get_histogram(images)
        train, valid, test = repeated_weighted_draw(hist, ratio_train, ratio_test, itrain, itest)
        
        _log_subject_eye(train, valid, test)
        
        # split into 3 folder
        _fill_file(images, ntpath.join(output_dir, "train.txt"), train[0])
        _fill_file(images, ntpath.join(output_dir, "valid.txt"), valid[0])
        _fill_file(images, ntpath.join(output_dir, "test.txt"), test[0])

split_dataset_source("dataset/source/classification_source_dataset.txt", "dataset/source/split/")


train : subject 73, eye 1153
valid : subject 33, eye 246
test  : subject 43, eye 248
total : subject 149, eye 1647



SPLIT INTO LEFT AND RIGHT -> COPY -> ANONYMISATION

In [ ]:
def _copy_files(files: list[str], output_dir):
    
    for file in files:
        shutil.copy(file, output_dir)

def _split_left_right(files: list[str]) -> tuple[list[str], list[str]]:
    left_pattern = "_L_"
    right_pattern = "_R_"
    
    def _is_contain_pattern(pattern, name):
        # safer to search only in the basename
        basename = ntpath.basename(name)
        return re.search(pattern, basename) is not None
        
    left = filter(partial(_is_contain_pattern, left_pattern), files)
    right = filter(partial(_is_contain_pattern, right_pattern), files)

    left = list(left)
    right = list(right)

    # a sort of safeguard (we can do better)
    if len(left) + len(right) != len(files):
        return
    
    return left, right

def anonymize(dir: str) -> dict[str, str]:
    
    # FIXME
    filenames = ""
    filenames = sorted(filenames)

    subject_ids: dict[str, int] = {}
    file_counters: dict[str, int] = defaultdict(int)

    res = {}

    for file in filenames:
        name = extract_subject_name(file)
        dirname = Path(file).parent

        if name not in subject_ids:
            subject_ids[name] = len(subject_ids)
        subject_id = subject_ids[name]

        file_idx = file_counters[name]
        file_counters[name] += 1

        ext = Path(file).suffix
        new_name = f"{subject_id:03d}{file_idx:03d}{ext}"

        res[file] = ntpath.join(dirname, new_name)

    return res

def split_dataset_images(sources_file, output_dir):
    
    # copy files into right or left folder
    Path(output_dir).mkdir(exist_ok=True, parents=True)
    
    left_path = ntpath.join(output_dir, "left")
    right_path = ntpath.join(output_dir, "right")
    
    Path(left_path).mkdir(exist_ok=True, parents=True)
    Path(right_path).mkdir(exist_ok=True, parents=True)
    
    with open(sources_file) as sources:
        
        filenames = sources.readlines()
        filenames = map(lambda s: s.strip(), filenames)
        
        # split left and right
        split = _split_left_right(filenames)
        
        if split is None:
            print(f"Failed to split {sources_file} into left and right")
            return
        
        left, right = split
        
        # copy files
        _copy_files(left, left_path)
        _copy_files(right, right_path)
        
        # anonymize(left_path)
        # anonymize(right_path)
        
        print(dict)
    
    
    # anonymise pour pouvoir publier le dataset
    
    # return
    
split_dataset_images("dataset/source/split/train.txt", "dataset/classification/train/")
split_dataset_images("dataset/source/split/valid.txt", "dataset/classification/valid/")
split_dataset_images("dataset/source/split/test.txt", "dataset/classification/test/")

{'Y:\\250326\\250326_ATM0044_R_1_HD_1\\png\\250326_ATM0044_R_1_HD_1_M0.png\n': 'Y:\\250326\\250326_ATM0044_R_1_HD_1\\png\\000000.png\n', 'Y:\\250326\\250326_ATM0044_R_2_HD_1\\png\\250326_ATM0044_R_2_HD_1_M0.png\n': 'Y:\\250326\\250326_ATM0044_R_2_HD_1\\png\\000001.png\n', 'Y:\\250326\\250326_ATM0044_R_HD_1\\png\\250326_ATM0044_R_HD_1_M0.png\n': 'Y:\\250326\\250326_ATM0044_R_HD_1\\png\\000002.png\n', 'Y:\\250326\\250326_AUZ0752_R_1_HD_1\\png\\250326_AUZ0752_R_1_HD_1_M0.png\n': 'Y:\\250326\\250326_AUZ0752_R_1_HD_1\\png\\001000.png\n', 'Y:\\250326\\250326_AUZ0752_R_2_HD_1\\png\\250326_AUZ0752_R_2_HD_1_M0.png\n': 'Y:\\250326\\250326_AUZ0752_R_2_HD_1\\png\\001001.png\n', 'Y:\\250326\\250326_AUZ0752_R_HD_1\\png\\250326_AUZ0752_R_HD_1_M0.png\n': 'Y:\\250326\\250326_AUZ0752_R_HD_1\\png\\001002.png\n', 'Y:\\250326\\250326_BOM0753_R_1_HD_1\\png\\250326_BOM0753_R_1_HD_1_M0.png\n': 'Y:\\250326\\250326_BOM0753_R_1_HD_1\\png\\002000.png\n', 'Y:\\250326\\250326_BOM0753_R_2_HD_1\\png\\250326_BOM0753_R